# LoRA fine-tuning for document extraction - GPU benchmark run

Runtime > Change runtime type > **T4 GPU** (free). Takes ~20-40 min for the 0.5B model.
This notebook produces the LLM rows of the results table; the CPU baselines (rules, CRF) are already in `results/preds`.

In [ ]:
!git clone https://github.com/MbarkyLyna/LoRA-Fine-Tuning-for-Document-Extraction.git repo
%cd repo
!pip install -q -e ".[llm]"
import torch; print(torch.cuda.get_device_name(0))

In [ ]:
# deterministic synthetic data (same seed => same files as the CPU benchmark)
!python -m docextract.synth --out data

In [ ]:
# 1) Benchmark the UNTUNED base model first (2-shot). LoRA has to beat this to be worth the effort.
SETS = "test_id_n00 test_id_n02 test_id_n05 test_ood_layout_n02 test_ood_lang_n02"
!python -m docextract.llm.infer --model Qwen/Qwen2.5-0.5B-Instruct --name qwen05b-fewshot2 --shots 2 --testset $SETS --limit 200

In [ ]:
# 2) LoRA fine-tune (rank 16, all linear layers). Logs trainable params, loss curve, time and peak memory.
!python -m docextract.llm.train_lora --model Qwen/Qwen2.5-0.5B-Instruct --out runs/qwen05b-lora --rank 16 --epochs 2

In [ ]:
# 3) Evaluate the adapter on the same documents
!python -m docextract.llm.infer --model Qwen/Qwen2.5-0.5B-Instruct --adapter runs/qwen05b-lora --name qwen05b-lora-r16 --testset $SETS --limit 200

In [ ]:
# 4) Same evaluator, all systems, intersection of documents -> results/results.md + figures
!python -m docextract.report
from IPython.display import Markdown, Image, display
display(Markdown(open('results/results.md').read()))
for f in ['noise_robustness','ood_generalisation','risk_coverage']:
    display(Image(f'results/figures/{f}.png'))

In [ ]:
# 5) Real-data transfer: LoRA model (trained on synthetic) on SROIE receipts
!bash scripts/prepare_sroie.sh
!python -m docextract.llm.infer --model Qwen/Qwen2.5-0.5B-Instruct --adapter runs/qwen05b-lora --name qwen05b-lora-r16 --testset sroie_test --out results/preds_real
!python -m docextract.real_sroie --evaluate

In [ ]:
# optional: LoRA rank/target sweep (adds ~1h on T4)
# !bash scripts/sweep_lora.sh

In [ ]:
# download predictions + adapter and commit them
!zip -qr llm_artifacts.zip results runs/qwen05b-lora
from google.colab import files; files.download('llm_artifacts.zip')